# Tutorial for DTM module

A document-term matrix (DTM) is the standard interface for analysis and information of document data, consisting of a list of term (unique token) counts per document arranged as a matrix. In many real-world text collections, most documents only contain a small fraction of all possible terms, resulting in a sparse matrix where the majority of values are zero—this allows for highly efficient storage and processing. The DTM is fundamental in text analysis because it transforms a collection of documents into a structured, numerical format suitable for computational analysis, enabling a wide range of analyses such as identifying common terms, comparing document similarity, and serving as input for machine learning models. By quantifying textual data, the DTM makes it possible to apply statistical and algorithmic techniques to extract insights from text. 

In this tutorial, we will understand what is a Document-Term Matrix, its applications, and what the `dtm` module can achieve, as we go through the process of using real texts to create our own DTM, and explore what can be be done with it.

## Preparing Your Documents

A document-term matrix is constructed by counting tokens to calculate the number of unique token forms -- or terms -- per document. This procedure requires the input documents to be already tokenised. In the cell below, we will import some test data and tokenise the documents.

Our data consists of 11 literary texts aquired from the [Project Gutenberg](https://www.gutenberg.org/) website, and saved locally in our `txt_files` directory.

Note that because we are loading whole novels, it may take a few minutes to perform the tokenization.

In [ ]:
# Import necessary modules
from lexos.io.loader import Loader
from lexos.tokenizer import Tokenizer

# Load the files
loader = Loader()
loader.load(["txt_files"])

# Tokenize the documents - this may take a while
tokenizer = Tokenizer(model="en_core_web_sm")
# texts = [text[0:5000] for text in loader.texts]  # Limit to first 5000 characters
docs = list(tokenizer.make_docs(loader.texts))

# Get labels for each document
labels = loader.names

If you do not need to access the NLP capabilities of Lexos' tokenizer, you can, alternatively, provide lists of tokens, one list for each document. This is especially useful if you already have pre-tokenised texts.

```python
docs = [
    ["token1", "token2", "token10"],
    ["another", "set", "of", "tokens", "token10"]
]
labels = ["Doc1", "Doc2"]
```

In fact, we probably want to do some pre-processing on the docs we just created in order to provide lists of tokens that do not include spaces, punctuation, or digits. Let's go ahead and do that.


In [ ]:
cleaned_docs = []
for doc in docs:
    tokens = [token for token in doc if not (token.is_space or token.is_punct or token.is_digit)]
    cleaned_docs.append(tokens)

## Understanding the Vectorizer

The `DTM` class creates a numerical matrix suitable for computational analysis by transforming your documents by using a vectorizer. A **vectorizer** is a tool that represents your documents in a matrix where each row represents a document and each column represents a unique term.

BY default, the Lexos DTM module uses Textacy's `Vectorizer` class to construct the matrix. Here's how it works in context:

- The `Vectorizer` scans all documents to build a vocabulary of unique terms (tokens).
- It then counts the occurrences of each term in each document, resulting in a sparse matrix where rows represent documents and columns represent terms.
- The `Vectorizer` can be customized to use different tokenization, filtering, or weighting schemes to better represent idiosyncracies in your data.

## Building the DTM

We're now ready to build our DTM from the 11 docs we have tokenised.

In [ ]:
# Import the DTM class
from lexos.dtm import DTM

# Initialize the DTM model
dtm = DTM()

# Build the DTM using the cleaned documents and labels
dtm(cleaned_docs, labels)

*Voilà*! We now have a DTM. Note that our labels above were created automatically by the `Loader` class. You can, of course, replace them with anything you want.

By default, `DTM` creates a document-term matrix where:

- Each column represents a single word (unigram/1-gram) from your corpus.
- Each cell contains the **raw count** of how many times that word appears in the document.
- If you want to include bigrams or trigrams (sequences of 2 or 3 words), or use proportions or TF-IDF scores, you can customize the vectorizer with additional parameters (see below).

For example, to get a table of proportions instead of raw counts, use `dtm.to_df(as_percent=True)`. To include bigrams, you would need to adjust the vectorizer settings (see the Textacy documentation for ngram support).

#### Customizing the Vectorizer

You can customize the vectorizer by passing keyword parameters when creating the `DTM` instance. For example, if you want to use TF-IDF weighting instead of raw counts, you can do so like this:

```python
dtm = DTM(tf_type="linear", idf_type="log")
```

You can also set the vectorizer's attributes directly, like so:

```python
dtm = DTM()
dtm.vectorizer.tf_type = "linear"
dtm.vectorizer.idf_type = "log"
```

Finally, you can pass keyword arguments to the vectorizer when calling the DTM class:

```python
dtm = DTM()
dtm(docs, labels, tf_type="linear", idf_type="log")
```

Which way you do it is a matter of personal preference, but one method may be preferable to another depending on the architecture of your application.

## Accessing the DTM's class methods and properties

The `shape` property returns the shape (number of documents, number of terms) of the DTM.

- `sorted_terms_list`: Returns a sorted list of all terms in the DTM.
- `sorted_term_counts`: Returns a sorted dictionary of terms and their total counts across all documents.

Let's check out the shape property first:

In [ ]:
print("DTM shape:", dtm.shape)

We can also inspect the terms and their counts in the DTM with the `sorted_terms_list` and `sorted_term_counts` properties. This allows you to quickly review which terms are present in your corpus vocabulary and how frequently they appear, providing valuable insights before further analysis or visualization.

In [ ]:
# Print samples from the sorted list of terms
unique_terms = dtm.sorted_terms_list
print("Sample terms:", unique_terms[:10])

# Print samples from the term counts mapping
counts = dtm.sorted_term_counts
sample_counts = dict(list(counts.items())[:10])
print("Sample term counts:", sample_counts)

We can also convert the DTM to a pandas DataFrame using the `to_df()` method. This allows for easier inspection, manipulation, and export of the document-term matrix, making it suitable for further analysis or visualization in pandas. We will also ensure that the rows indicate documents and columns indicate terms by transposing the DataFrame.

In [ ]:
df = dtm.to_df()
df

### Customizing the DataFrame Output

You can tailor the output of your document-term matrix DataFrame using several options in the `to_df()` method:

- **Sort by a specific column:**  
    Use the `by` parameter to sort the DataFrame by a particular document label (e.g., `by="Doc2"`). By default, the DataFrame is sorted by the first label.

- **Control sort order:**  
    Set `ascending=False` to sort in descending order (largest to smallest values). The default is `ascending=True` for ascending order.

- **Display percentages instead of raw counts:**  
    Set `as_percent=True` to show term frequencies as percentages of the total terms in each document, rather than raw counts. This is useful for comparing documents of different lengths.

- **Adjust decimal precision:**  
    Use the `rounding` parameter to specify the number of decimal places for percentages (e.g., `rounding=2` for two decimals). The default is 1 decimal place.

- **Add row statistics:**  
    Include summary statistics for each term across all documents by setting `sum=True` (total count), `mean=True` (average count), or `median=True` (median count). These columns help you quickly identify the most common or distinctive terms.

- **Transpose the matrix:**  
    Set `transpose=True` to swap rows and columns, so that documents become columns and terms become rows. This can be helpful for certain types of analysis or visualization.

Once the dataframe has been created, it can be further manipulated using the pandas interface.

<a id="62-example-with-percentages-with-totals-and-two-decimals"></a>
#### 6.2 Example with percentages with totals and two decimals

This example demonstrates how to convert the document-term matrix to a DataFrame that displays term frequencies as percentages, rounded to two decimal places, and includes a total count column for each term. This format makes it easy to compare term usage across documents of different lengths and quickly identify the most frequent terms in the corpus.

In [ ]:
df_pct = dtm.to_df(
    as_percent=True,
    rounding=2,
    sum=True
)
df_pct = df_pct.transpose()
df_pct

<a id="7-visualizing-the-dtm"></a>
## 7. Visualizing the DTM
The DataFrame output from `to_df()` makes it easy to visualize your document-term matrix using standard pandas plotting methods. Let's generate two simple graphs:

The bar chart below displays the top 20 most frequent terms across our entire corpus of classical texts. Each bar represents a term, and its height corresponds to the total number of times that term appears in all documents combined.

In [ ]:
# Get the first 20 rows of the DTM as a DataFrame sorted by sum
df = dtm.to_df(sum=True, by="Total", ascending=False)[0:20]

# Plot the DataFrame
df.Total.plot(
    kind="bar",
    title="Top 20 Most Frequent Terms",
    xlabel="Terms",
    ylabel="Frequency"
)

We can even use Lexos' own `cloud` module to generate a word cloud of the most frequent terms in our corpus. In this visualization, each word's size is proportional to its overall frequency in the corpus—the larger the word appears, the more often it occurs.

In [ ]:
from lexos.visualization.cloud import wordcloud

wordcloud(dtm)

In addition to these basic visualizations, the Lexos API provides built-in tools for more advanced visualizations. For more details and examples, refer to the [Visualization page](#) or the relevant Lexos API documentation. These tools help you explore and present your text data more effectively.

<a id="8-applications-and-experimentation"></a>
## 8. Applications and Experimentation
By building and analyzing a Document-Term Matrix (DTM), you unlock a wide range of real-world applications in text analysis and natural language processing. Here are some practical ways to apply these techniques:

- **Text Classification:** Use the DTM as input features for machine learning models to classify documents by topic, sentiment, or author.
- **Topic Modeling:** Identify underlying themes in large corpora by applying algorithms like Latent Dirichlet Allocation (LDA) to the DTM.
- **Authorship Attribution:** Compare term usage patterns across documents to attribute anonymous texts to likely authors.
- **Plagiarism Detection:** Measure document similarity using DTM-based metrics to detect copied or closely paraphrased content.
- **Trend Analysis:** Track the frequency of specific terms or topics over time to analyze trends in news, literature, or social media.
- **Keyword Extraction:** Identify the most distinctive or frequent terms in a set of documents for summarization or search optimization.
- **Corpus Exploration:** Visualize and explore large text collections to uncover patterns, outliers, or clusters of related documents.

By converting raw text into structured, quantitative data with the DTM, you can apply statistical, machine learning, and visualization techniques to gain actionable insights from textual information in fields such as digital humanities, marketing, social sciences, and beyond.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.cluster import KMeans

# Example applications using the loaded texts and DTM

# 1. Text Classification (e.g., classify by author using term frequencies)

# For demonstration, let's assign a dummy author label to each document
authors = [
    "Carroll", "Stoker", "Shelley", "Kafka", "Austen", "Shakespeare",
    "Doyle", "Fitzgerald", "Wilde", "Stevenson", "Nietzsche"
]
# If there are more documents, extend or repeat author labels as needed
y = authors[:len(labels)]

X = dtm.doc_term_matrix.toarray()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = MultinomialNB().fit(X_train, y_train)
print("Text Classification accuracy:", clf.score(X_test, y_test))

In [ ]:
# 2. Topic Modeling (LDA)

lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(X)
for idx, topic in enumerate(lda.components_):
    top_terms = [dtm.sorted_terms_list[i] for i in topic.argsort()[-5:][::-1]]
    print(f"Topic {idx+1} top terms:", top_terms)


In [ ]:
# 3. Authorship Attribution (find most likely author for a new text)
test_text = nlp("It was a bright cold day in April, and the clocks were striking thirteen.")
test_doc = nlp.make_doc(" ".join([
    token.text for token in test_text
    if not (token.is_space or token.is_punct or token.is_stop)
]))
test_vec = dtm.vectorizer.transform([test_doc])
predicted_author = clf.predict(test_vec.toarray())[0]
print("Predicted author for test text:", predicted_author)

In [ ]:
# 4. Plagiarism Detection (cosine similarity between documents)

similarity_matrix = cosine_similarity(X)
np.fill_diagonal(similarity_matrix, 0)
most_similar = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
print(f"Most similar documents: {labels[most_similar[0]]} and {labels[most_similar[1]]} (similarity={similarity_matrix[most_similar]:.2f})")

In [ ]:
# 5. Trend Analysis (track frequency of a term across documents)
term = "said"
if term in dtm.sorted_terms_list:
    idx = dtm.sorted_terms_list.index(term)
    term_freqs = X[:, idx]
    for label, freq in zip(labels, term_freqs):
        print(f"Frequency of '{term}' in '{label}': {freq}")
else:
    print(f"Term '{term}' not found in vocabulary.")

In [ ]:
# 6. Keyword Extraction (top terms in a document)
doc_idx = 0
top_terms_idx = X[doc_idx].argsort()[-5:][::-1]
top_terms = [dtm.sorted_terms_list[i] for i in top_terms_idx]
print(f"Top terms in '{labels[doc_idx]}':", top_terms)

In [ ]:
# 7. Corpus Exploration (cluster documents by similarity)
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
for i, label in enumerate(labels):
    print(f"Document '{label}' assigned to cluster {kmeans.labels_[i]}")